In [5]:
# Set up paths, output subfolder, and strict/exclusive-assignment parameters for Nanopore S1–S4 stacked coverage analysis.

from pathlib import Path
import gzip
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import edlib
import os
from os import mkdir

# --- REQUIRED: project root ---
PROJECT_DIR = Path("")
DATA_DIR = PROJECT_DIR / "data"
os.makedirs(DATA_DIR, exist_ok=True)
OUTPUT_ROOT = PROJECT_DIR / "output"
os.makedirs(OUTPUT_ROOT, exist_ok=True)
# --- REQUIRED: user-named output subfolder ---
OUTPUT_SUBDIR_NAME = "stacked_coverage_exclusive"
OUTPUT_DIR = OUTPUT_ROOT / OUTPUT_SUBDIR_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- References expected in DATA_DIR ---
REF_NAMES = ["S1", "S2", "S3", "S4"]
REF_FASTA = {r: DATA_DIR / f"{r}_Amplicon.fasta" for r in REF_NAMES}

# --- STRICT thresholds ---
MIN_IDENTITY_STRICT = 0.50
MAX_LEN_DIFF_STRICT = 20

# --- Exclusive assignment ambiguity filters (set to 0 to disable) ---
MIN_EDITDIST_MARGIN = 5        # require (2nd_best_ed - best_ed) >= this
MIN_IDENTITY_MARGIN = 0.002    # require (best_ident - 2nd_best_ident) >= this

# --- Plot settings ---
plt.rcParams["font.family"] = "Arial"

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


PROJECT_DIR: .
DATA_DIR: data
OUTPUT_DIR: output/stacked_coverage_exclusive


In [6]:
# Discover FASTQ/FASTQ.GZ files and define a robust FASTQ reader generator.

def list_fastq_files(data_dir: Path):
    exts = (".fastq", ".fq", ".fastq.gz", ".fq.gz")
    return sorted([p for p in data_dir.iterdir() if p.is_file() and p.name.lower().endswith(exts)])

def read_fastq(path: Path):
    opener = gzip.open if path.name.lower().endswith(".gz") else open
    with opener(path, "rt") as f:
        while True:
            header = f.readline().rstrip("\n")
            if not header:
                break
            seq = f.readline().rstrip("\n")
            plus = f.readline().rstrip("\n")
            qual = f.readline().rstrip("\n")
            if not qual:
                break
            yield header, seq, plus, qual

fastq_files = list_fastq_files(DATA_DIR)
print(f"Found {len(fastq_files)} FASTQ files:")
for p in fastq_files:
    print(" -", p.name)


Found 5 FASTQ files:
 - 3TKLDP_1_Smix_P1.fastq
 - 3TKLDP_2_Smix_P2.fastq
 - 3TKLDP_3_Smix_P3.fastq
 - 3TKLDP_4_Smix_P4.fastq
 - 3TKLDP_5_Smix_P5.fastq


In [7]:
# Load S1–S4 reference sequences from FASTA files in the data folder.

def load_single_fasta(path: Path):
    header = None
    seq_lines = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if header is None:
                    header = line[1:].strip()
                else:
                    break
            else:
                seq_lines.append(line)
    if header is None:
        header = path.stem
    return header, "".join(seq_lines).upper()

ref_seqs = {}
for r in REF_NAMES:
    p = REF_FASTA[r]
    if not p.exists():
        raise FileNotFoundError(f"Missing reference FASTA: {p}")
    hdr, seq = load_single_fasta(p)
    ref_seqs[r] = seq
    print(f"{r}: {p.name} | len={len(seq)} | header='{hdr}'")


S1: S1_Amplicon.fasta | len=1694 | header='S1  (1694 bp)'
S2: S2_Amplicon.fasta | len=1694 | header='S2  (1694 bp)'
S3: S3_Amplicon.fasta | len=1694 | header='S3  (1694 bp)'
S4: S4_Amplicon.fasta | len=1694 | header='S4  (1694 bp)'


In [8]:
# Define global-identity scoring and strict gating used by exclusive assignment.

def global_identity_from_ed(ed: int, qlen: int, rlen: int) -> float:
    denom = max(qlen, rlen)
    return 1.0 - (ed / denom if denom > 0 else 0.0)

def len_gate(seq: str, ref: str, max_len_diff: int) -> bool:
    return abs(len(seq) - len(ref)) <= max_len_diff

def best_two_refs(seq: str, ref_seqs: dict, ref_names):
    """
    Return best and second-best refs ranked by edit distance (global NW).
    Output: (best_r, best_ident, best_ed, second_r, second_ident, second_ed)
    """
    scored = []
    for r in ref_names:
        ref = ref_seqs[r]
        res = edlib.align(seq, ref, mode="NW", task="distance")
        ed = res["editDistance"]
        ident = global_identity_from_ed(ed, len(seq), len(ref))
        scored.append((r, ident, ed))
    scored.sort(key=lambda x: x[2])
    (b_r, b_ident, b_ed) = scored[0]
    (s_r, s_ident, s_ed) = scored[1]
    return b_r, b_ident, b_ed, s_r, s_ident, s_ed


In [9]:
# Build reference-coordinate base-count pileups (A/C/G/T/N) from global alignments and convert to coverage per position.

def parse_cigar(cigar: str):
    num = ""
    for ch in cigar:
        if ch.isdigit():
            num += ch
        else:
            yield int(num), ch
            num = ""

def build_ref_aligned_counts(seqs, ref_seq: str):
    """
    Align each seq to ref_seq (edlib NW, task=path) and accumulate base counts per ref position.
    Returns: list[Counter] length = len(ref_seq), each Counter over A/C/G/T/N.
    """
    ref_len = len(ref_seq)
    counts = [Counter() for _ in range(ref_len)]

    for seq in seqs:
        res = edlib.align(seq, ref_seq, mode="NW", task="path")
        cigar = res["cigar"]
        q_pos, r_pos = 0, 0

        for length, op in parse_cigar(cigar):
            if op in ("M", "=", "X"):
                for _ in range(length):
                    if 0 <= r_pos < ref_len and 0 <= q_pos < len(seq):
                        b = seq[q_pos].upper()
                        if b in "ACGTN":
                            counts[r_pos][b] += 1
                    q_pos += 1
                    r_pos += 1
            elif op == "I":
                q_pos += length
            elif op == "D":
                r_pos += length

    return counts

def counts_to_coverage(counts):
    return np.array([sum(c.values()) for c in counts], dtype=int)


In [10]:
# Exclusively assign each read to S1/S2/S3/S4 under strict thresholds, dropping ambiguous near-ties.

strict_reads_excl = {r: {} for r in REF_NAMES}
summary_rows = []

for fq in fastq_files:
    sample = fq.stem.replace(".fastq", "").replace(".fq", "")
    for r in REF_NAMES:
        strict_reads_excl[r][sample] = []

    n_total = 0
    n_assigned = 0
    n_ambiguous = 0
    n_failed = 0

    for header, seq, plus, qual in read_fastq(fq):
        n_total += 1
        seq = seq.upper()

        # Fast length gate: if seq is outside the window for ALL refs, skip
        if not any(len_gate(seq, ref_seqs[r], MAX_LEN_DIFF_STRICT) for r in REF_NAMES):
            n_failed += 1
            continue

        best_r, best_ident, best_ed, second_r, second_ident, second_ed = best_two_refs(seq, ref_seqs, REF_NAMES)

        # Strict on BEST only
        if (not len_gate(seq, ref_seqs[best_r], MAX_LEN_DIFF_STRICT)) or (best_ident < MIN_IDENTITY_STRICT):
            n_failed += 1
            continue

        # Ambiguity margin (optional)
        ed_margin = second_ed - best_ed
        ident_margin = best_ident - second_ident
        if (MIN_EDITDIST_MARGIN > 0 and ed_margin < MIN_EDITDIST_MARGIN) or (MIN_IDENTITY_MARGIN > 0 and ident_margin < MIN_IDENTITY_MARGIN):
            n_ambiguous += 1
            continue

        strict_reads_excl[best_r][sample].append(seq)
        n_assigned += 1

    row = {
        "sample": sample,
        "total_reads": n_total,
        "strict_assigned": n_assigned,
        "ambiguous_dropped": n_ambiguous,
        "failed_strict": n_failed,
    }
    for r in REF_NAMES:
        row[f"{r}_strict"] = len(strict_reads_excl[r][sample])

    summary_rows.append(row)

    print(f"[{sample}] total={n_total:,} assigned={n_assigned:,} ambiguous={n_ambiguous:,} failed={n_failed:,} | " +
          " ".join([f"{r}:{row[f'{r}_strict']:,}" for r in REF_NAMES]))

df_assign = pd.DataFrame(summary_rows)
df_assign.to_csv(OUTPUT_DIR / "exclusive_assignment_summary_STRICT.csv", index=False)
df_assign


[3TKLDP_1_Smix_P1] total=10,000 assigned=3,325 ambiguous=0 failed=6,675 | S1:664 S2:663 S3:952 S4:1,046
[3TKLDP_2_Smix_P2] total=10,000 assigned=3,400 ambiguous=0 failed=6,600 | S1:615 S2:730 S3:872 S4:1,183
[3TKLDP_3_Smix_P3] total=10,000 assigned=3,254 ambiguous=0 failed=6,746 | S1:785 S2:963 S3:793 S4:713
[3TKLDP_4_Smix_P4] total=10,000 assigned=3,211 ambiguous=0 failed=6,789 | S1:619 S2:770 S3:829 S4:993
[3TKLDP_5_Smix_P5] total=10,000 assigned=3,331 ambiguous=0 failed=6,669 | S1:705 S2:681 S3:880 S4:1,065


,sample,total_reads,strict_assigned,ambiguous_dropped,failed_strict,S1_strict,S2_strict,S3_strict,S4_strict
0,3TKLDP_1_Smix_P1,10000,3325,0,6675,664,663,952,1046
1,3TKLDP_2_Smix_P2,10000,3400,0,6600,615,730,872,1183
2,3TKLDP_3_Smix_P3,10000,3254,0,6746,785,963,793,713
3,3TKLDP_4_Smix_P4,10000,3211,0,6789,619,770,829,993
4,3TKLDP_5_Smix_P5,10000,3331,0,6669,705,681,880,1065


In [11]:
# Compute per-position coverage for each (sample, reference) from exclusive strict reads using reference-coordinate pileups.

coverage_tables = {r: {} for r in REF_NAMES}

for fq in fastq_files:
    sample = fq.stem.replace(".fastq", "").replace(".fq", "")

    for r in REF_NAMES:
        ref_seq = ref_seqs[r]
        ref_len = len(ref_seq)
        seqs = strict_reads_excl[r].get(sample, [])

        if len(seqs) == 0:
            continue

        counts = build_ref_aligned_counts(seqs, ref_seq)
        cov = counts_to_coverage(counts)

        df = pd.DataFrame({
            "position": np.arange(1, ref_len + 1),
            "coverage": cov
        })
        coverage_tables[r][sample] = df

        out_csv = OUTPUT_DIR / f"{sample}_{r}_coverage_STRICT_exclusive.csv"
        df.to_csv(out_csv, index=False)

print("Saved per-reference coverage CSVs (where reads existed).")


Saved per-reference coverage CSVs (where reads existed).


In [12]:
# Create and save per-sample stacked coverage plots (S1–S4) and merged coverage tables.

def make_stacked_plot(x, stacks, sample_name, out_svg: Path):
    plt.figure(figsize=(12, 4))
    plt.stackplot(x, *stacks, labels=REF_NAMES)
    plt.xlabel("Position (bp)")
    plt.ylabel("Coverage (reads)")
    plt.title(f"Stacked coverage (STRICT, exclusive) – {sample_name}")
    plt.legend(loc="upper right", fontsize=8)
    plt.tight_layout()
    plt.savefig(out_svg, format="svg")
    plt.close()

for fq in fastq_files:
    sample = fq.stem.replace(".fastq", "").replace(".fq", "")

    merged = None
    for r in REF_NAMES:
        df = coverage_tables[r].get(sample, None)
        if df is None:
            continue
        df = df.rename(columns={"coverage": f"cov_{r}"})
        merged = df if merged is None else merged.merge(df, on="position", how="outer")

    if merged is None:
        print(f"[{sample}] No exclusive strict coverage for any ref; skipping.")
        continue

    for r in REF_NAMES:
        col = f"cov_{r}"
        if col not in merged.columns:
            merged[col] = 0

    merged = merged.sort_values("position").reset_index(drop=True)

    merged_csv = OUTPUT_DIR / f"{sample}_S1S4_stacked_coverage_STRICT_exclusive.csv"
    merged.to_csv(merged_csv, index=False)

    x = merged["position"].to_numpy()
    stacks = [merged[f"cov_{r}"].to_numpy() for r in REF_NAMES]

    out_svg = OUTPUT_DIR / f"{sample}_S1S4_stacked_coverage_STRICT_exclusive.svg"
    make_stacked_plot(x, stacks, sample, out_svg)

    print(f"[{sample}] Saved: {merged_csv.name} + {out_svg.name}")


[3TKLDP_1_Smix_P1] Saved: 3TKLDP_1_Smix_P1_S1S4_stacked_coverage_STRICT_exclusive.csv + 3TKLDP_1_Smix_P1_S1S4_stacked_coverage_STRICT_exclusive.svg
[3TKLDP_2_Smix_P2] Saved: 3TKLDP_2_Smix_P2_S1S4_stacked_coverage_STRICT_exclusive.csv + 3TKLDP_2_Smix_P2_S1S4_stacked_coverage_STRICT_exclusive.svg
[3TKLDP_3_Smix_P3] Saved: 3TKLDP_3_Smix_P3_S1S4_stacked_coverage_STRICT_exclusive.csv + 3TKLDP_3_Smix_P3_S1S4_stacked_coverage_STRICT_exclusive.svg
[3TKLDP_4_Smix_P4] Saved: 3TKLDP_4_Smix_P4_S1S4_stacked_coverage_STRICT_exclusive.csv + 3TKLDP_4_Smix_P4_S1S4_stacked_coverage_STRICT_exclusive.svg
[3TKLDP_5_Smix_P5] Saved: 3TKLDP_5_Smix_P5_S1S4_stacked_coverage_STRICT_exclusive.csv + 3TKLDP_5_Smix_P5_S1S4_stacked_coverage_STRICT_exclusive.svg


In [13]:
# Summarize total exclusive-assigned read counts per reference for each sample as a quick sanity check.

df = pd.read_csv(OUTPUT_DIR / "exclusive_assignment_summary_STRICT.csv")
cols = ["sample"] + [f"{r}_strict" for r in REF_NAMES]
df_small = df[cols].copy()
df_small


,sample,S1_strict,S2_strict,S3_strict,S4_strict
0,3TKLDP_1_Smix_P1,664,663,952,1046
1,3TKLDP_2_Smix_P2,615,730,872,1183
2,3TKLDP_3_Smix_P3,785,963,793,713
3,3TKLDP_4_Smix_P4,619,770,829,993
4,3TKLDP_5_Smix_P5,705,681,880,1065
